# Sen1Floods11 — FCN-ResNet50 training reproduction


The actual FCN training computation is kept unchanged:
- FCN-ResNet50
- 2-channel Sentinel-1 input
- GroupNorm conversion
- weighted cross-entropy
- AdamW
- released-code cosine scheduler behavior
- original augmentation/normalization
- original batch sizes and metrics


`D:\UNI\THESIS\datasets\Sen1Floods11\v1.1`

> **PATCHED 2026-10-01:** permanent-water target validation, nearest-neighbor mask resize, CPU preflight, and safe CUDA target handling.

> **PATCHED V2 2026-10-01:** non-finite values in JRC permanent-water target masks are converted to `255` (ignore) and reported during preflight.


>NOTE: <p style="color:green">**THIS IS VERSION 1**</p>

## 1. Local runtime setup

Open this notebook in the **remote-connected VS Code window** and select a Python kernel installed on the Windows PC.

The first cell installs only notebook/data dependencies. It deliberately does **not** install or replace PyTorch, because your CUDA-enabled PyTorch installation should be managed separately.

In [ ]:
%pip install -q rasterio tqdm ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import os
import sys
import csv
import json
import shutil           # high-level file operations (shell utilities)
import subprocess       # run external commands or shell commands
import platform         # informations about the OS and the machine
import random

import numpy as np
import rasterio
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by this Python kernel. "
        "In VS Code, make sure the selected REMOTE Windows Python environment "
        "has a CUDA-enabled PyTorch build installed."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Notebook is running on:", platform.node())
print("OS:", platform.platform())

Python: 3.12.10
PyTorch: 2.14.0+cu130
Torchvision: 0.29.0+cu130
CUDA runtime: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3070
Notebook is running on: SI02T12
OS: Windows-11-10.0.26200-SP0


#### Local Paths

In [ ]:
# ------------------------------------------------------------------
# LOCAL WINDOWS PATHS
# ------------------------------------------------------------------
# Change only this line if you later move the Sen1Floods11 v1.1 folder.
DATASET_ROOT = Path(r"D:\UNI\THESIS\datasets\Sen1Floods11\v1.1")

DATA_DIR = DATASET_ROOT / "data"
SPLIT_DIR = DATASET_ROOT / "splits"

# Keep generated experiment outputs separate from the downloaded dataset.
OUTPUT_ROOT = Path(r"D:\UNI\THESIS\experiments\sen1floods11\runs")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
RESULTS_ROOT = OUTPUT_ROOT / "results"
LOGS_ROOT = OUTPUT_ROOT / "logs"

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {DATASET_ROOT}\n"
        "Edit DATASET_ROOT above if the dataset was moved."
    )

for p in (CHECKPOINT_ROOT, RESULTS_ROOT, LOGS_ROOT):
    p.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Data directory:", DATA_DIR)
print("Split directory:", SPLIT_DIR)
print("Persistent outputs:", OUTPUT_ROOT)

Dataset root: D:\UNI\THESIS\datasets\v1.1
Data directory: D:\UNI\THESIS\datasets\v1.1\data
Split directory: D:\UNI\THESIS\datasets\v1.1\splits
Persistent outputs: D:\UNI\THESIS\Sen1Floods11_runs


## 2. Choose one experiment

Run **one experiment at a time**. Change only `EXPERIMENT`, then rerun the notebook from this cell downward.

Published training durations:

- hand-labeled: 100 epochs
- S1 weak: 3 epochs
- S2 weak: 200 epochs
- permanent water: 200 epochs

In [ ]:
EXPERIMENT = "permanent_water"
# choices:
# "hand_labeled"
# "s1_weak"
# "s2_weak"
# "permanent_water"

EXPERIMENTS = {
    "hand_labeled": {
        "epochs": 100,
        "train_kind": "csv_hand",
    },
    "s1_weak": {
        "epochs": 3,
        "train_kind": "dir_s1_weak",
    },
    "s2_weak": {
        "epochs": 200,
        "train_kind": "dir_s2_weak",
    },
    "permanent_water": {
        "epochs": 200,
        "train_kind": "csv_perm",
    },
}

if EXPERIMENT not in EXPERIMENTS:
    raise ValueError(f"Unknown experiment: {EXPERIMENT}")

LR = 5e-4
EPOCHS = EXPERIMENTS[EXPERIMENT]["epochs"]

# Released notebook behavior: validate once per epoch.
EPOCHS_PER_UPDATE = 1

# If True and resume_checkpoint.pt exists for this experiment,
# training continues from the last COMPLETED epoch.
RESUME_IF_AVAILABLE = True

# Validate all raw masks on CPU before starting/resuming CUDA training.
PREFLIGHT_VALIDATE_LABELS = True

RUNNAME = f"Sen1Floods11_{EXPERIMENT}"
CHECKPOINT_DIR = CHECKPOINT_ROOT / EXPERIMENT
RESULTS_DIR = RESULTS_ROOT / EXPERIMENT
LOG_DIR = LOGS_ROOT / EXPERIMENT

for p in (CHECKPOINT_DIR, RESULTS_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = CHECKPOINT_DIR / "resume_checkpoint.pt"

print("Experiment:", EXPERIMENT)
print("Epochs:", EPOCHS)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Resume checkpoint:", RESUME_CHECKPOINT)

Experiment: permanent_water
Epochs: 200
Checkpoint directory: D:\UNI\THESIS\Sen1Floods11_runs\checkpoints\permanent_water
Resume checkpoint: D:\UNI\THESIS\Sen1Floods11_runs\checkpoints\permanent_water\resume_checkpoint.pt


## 3. Use the already-downloaded local dataset

No Colab download is performed.

The notebook reads directly from:

`D:\UNI\THESIS\datasets\Sen1Floods11\v1.1`

Expected layout:

```text
v1.1/
├── data/
│   ├── flood_events/
│   │   ├── HandLabeled/
│   │   └── WeaklyLabeled/
│   └── perm_water/
└── splits/
    ├── flood_handlabeled/
    └── perm_water/
```

The weak training sets do not have their own train/validation/test CSVs. Their TIFF pairs are constructed from the weak-label directories, as in the historical training notebook.

In [ ]:
FLOOD_SPLIT_DIR = SPLIT_DIR / "flood_handlabeled"
PERM_SPLIT_DIR = SPLIT_DIR / "perm_water"

HAND_DIR = DATA_DIR / "flood_events" / "HandLabeled"
WEAK_DIR = DATA_DIR / "flood_events" / "WeaklyLabeled"
PERM_DIR = DATA_DIR / "perm_water"

S1_HAND_DIR = HAND_DIR / "S1Hand"
LABEL_HAND_DIR = HAND_DIR / "LabelHand"

S1_WEAK_DIR = WEAK_DIR / "S1Weak"
S1_WEAK_LABEL_DIR = WEAK_DIR / "S1OtsuLabelWeak"
S2_WEAK_LABEL_DIR = WEAK_DIR / "S2IndexLabelWeak"

S1_PERM_DIR = PERM_DIR / "S1Perm"
JRC_PERM_DIR = PERM_DIR / "JRCPerm"

FLOOD_TRAIN_CSV = FLOOD_SPLIT_DIR / "flood_train_data.csv"
FLOOD_VALID_CSV = FLOOD_SPLIT_DIR / "flood_valid_data.csv"
FLOOD_TEST_CSV = FLOOD_SPLIT_DIR / "flood_test_data.csv"
FLOOD_BOLIVIA_CSV = FLOOD_SPLIT_DIR / "flood_bolivia_data.csv"

PERM_TRAIN_CSV = PERM_SPLIT_DIR / "permanent_water_train_data.csv"
PERM_VALID_CSV = PERM_SPLIT_DIR / "permanent_water_validation_data.csv"
PERM_TEST_CSV = PERM_SPLIT_DIR / "permanent_water_test_data.csv"

print("Using local Sen1Floods11 data.")

Using local Sen1Floods11 data.


In [ ]:
required_base_paths = [
    DATA_DIR,
    SPLIT_DIR,
    FLOOD_SPLIT_DIR,
    S1_HAND_DIR,
    LABEL_HAND_DIR,
]

missing_base = [p for p in required_base_paths if not p.exists()]
if missing_base:
    print("Missing required local paths:")
    for p in missing_base:
        print("  ", p)
    raise FileNotFoundError(
        "The local Sen1Floods11 folder does not match the expected v1.1 layout."
    )

print("Base dataset folders found successfully.")

Base dataset folders found successfully.


## 4. Inspect what was downloaded

This cell is intentionally strict. It prevents training from silently proceeding with empty or mismatched directories.

In [ ]:
def tif_count(path):
    return len(list(Path(path).glob("*.tif")))

print("S1Hand:", tif_count(S1_HAND_DIR))
print("LabelHand:", tif_count(LABEL_HAND_DIR))

if tif_count(S1_HAND_DIR) != 446 or tif_count(LABEL_HAND_DIR) != 446:
    raise RuntimeError("Expected 446 S1Hand and 446 LabelHand TIFFs.")

for csv_path in [
    FLOOD_TRAIN_CSV,
    FLOOD_VALID_CSV,
    FLOOD_TEST_CSV,
    FLOOD_BOLIVIA_CSV
]:
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

if EXPERIMENT == "s1_weak":
    n_images = tif_count(S1_WEAK_DIR)
    n_labels = tif_count(S1_WEAK_LABEL_DIR)

    print("S1Weak:", n_images)
    print("S1OtsuLabelWeak:", n_labels)

    if n_images != n_labels:
        raise RuntimeError(
            f"S1 weak image/label mismatch: {n_images} images vs {n_labels} labels."
        )

    if n_images not in (4384, 4385):
        raise RuntimeError(
            f"Unexpected S1 weak dataset size: {n_images}. Expected 4384 or 4385."
        )

elif EXPERIMENT == "s2_weak":
    n_images = tif_count(S1_WEAK_DIR)
    n_labels = tif_count(S2_WEAK_LABEL_DIR)

    print("S1Weak:", n_images)
    print("S2IndexLabelWeak:", n_labels)

    if n_images != n_labels:
        raise RuntimeError(
            f"S2 weak image/label mismatch: {n_images} images vs {n_labels} labels."
        )

    if n_images not in (4384, 4385):
        raise RuntimeError(
            f"Unexpected S2 weak dataset size: {n_images}. Expected 4384 or 4385."
        )

elif EXPERIMENT == "permanent_water":
    print("S1Perm:", tif_count(S1_PERM_DIR))
    print("JRCPerm:", tif_count(JRC_PERM_DIR))

    for csv_path in [
        PERM_TRAIN_CSV,
        PERM_VALID_CSV,
        PERM_TEST_CSV
    ]:
        if not csv_path.exists():
            raise FileNotFoundError(csv_path)

S1Hand: 446
LabelHand: 446
S1Perm: 814
JRCPerm: 814


## 5. Original preprocessing / augmentation

The functions below are kept from the released `Train.ipynb`:

- random 256×256 crop from each 512×512 training chip
- horizontal/vertical flips
- normalization with `[0.6851, 0.5235]` and `[0.0820, 0.1102]`
- validation/test split into four 256×256 quadrants

In [ ]:
from torchvision import transforms
import torchvision.transforms.functional as F
import random
from PIL import Image

VALID_TARGET_VALUES = {0, 1, 255}


def sanitize_label_array(
    arr,
    source="<in-memory>",
    *,
    report_nonfinite=False,
    return_nonfinite_count=False,
):
    """
    Canonical Sen1Floods11 binary target:
      NaN/Inf -> 255 (ignore / no usable target)
      -1      -> 255 (ignore)
       0      -> class 0
       1      -> class 1
       255    -> ignore

    Any other finite class value is rejected on CPU.

    The original paper defines uncertain/non-usable permanent-water pixels as
    ignored pixels. The public v1.1 JRCPerm GeoTIFFs can also contain non-finite
    raster values, so those pixels are conservatively treated as ignore rather
    than being converted into class 0 or class 1.
    """
    arr = np.asarray(arr).squeeze()

    if arr.ndim != 2:
        raise ValueError(
            f"{source}: expected one 2-D label mask, got shape {np.asarray(arr).shape}"
        )

    # Handle raster NoData/non-finite pixels before integer-class validation.
    nonfinite = ~np.isfinite(arr)
    nonfinite_count = int(nonfinite.sum())

    if nonfinite_count:
        arr = arr.copy()
        arr[nonfinite] = 255

        if report_nonfinite:
            total = int(arr.size)
            pct = 100.0 * nonfinite_count / total
            print(
                f"WARNING: {source}: {nonfinite_count}/{total} "
                f"({pct:.4f}%) NaN/Inf label pixels -> ignore_index 255"
            )

    rounded = np.rint(arr)
    if not np.allclose(arr, rounded, rtol=0, atol=1e-6):
        vals = np.unique(arr)
        raise ValueError(
            f"{source}: label contains non-integer finite class values. "
            f"First values: {vals[:30].tolist()}"
        )

    arr = rounded.astype(np.int64, copy=False).copy()
    arr[arr == -1] = 255

    values = set(np.unique(arr).tolist())
    invalid = sorted(values - VALID_TARGET_VALUES)
    if invalid:
        raise ValueError(
            f"{source}: invalid finite target class IDs {invalid}. "
            f"Allowed values after canonicalization are 0, 1 and 255."
        )

    clean = arr.astype(np.uint8, copy=False)

    if return_nonfinite_count:
        return clean, nonfinite_count

    return clean


def mask_to_pil(mask, source="<in-memory>"):
    arr = sanitize_label_array(mask, source=source)
    return Image.fromarray(arr, mode="L")


def mask_pil_to_long(mask, source="<preprocessed-mask>"):
    arr = np.array(mask, dtype=np.uint8, copy=True)
    arr = sanitize_label_array(arr, source=source)
    return torch.from_numpy(arr.astype(np.int64, copy=False))


def processAndAugment(data):
    (x, y) = data
    im = x.copy()

    im1 = Image.fromarray(im[0])
    im2 = Image.fromarray(im[1])
    label = mask_to_pil(y)

    i, j, h, w = transforms.RandomCrop.get_params(im1, (256, 256))

    im1 = F.crop(im1, i, j, h, w)
    im2 = F.crop(im2, i, j, h, w)
    label = F.crop(label, i, j, h, w)

    if random.random() > 0.5:
        im1 = F.hflip(im1)
        im2 = F.hflip(im2)
        label = F.hflip(label)

    if random.random() > 0.5:
        im1 = F.vflip(im1)
        im2 = F.vflip(im2)
        label = F.vflip(label)

    norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])

    im = torch.stack([
        transforms.ToTensor()(im1).squeeze(),
        transforms.ToTensor()(im2).squeeze(),
    ])
    im = norm(im)

    # Semantic class IDs are converted directly to integer targets.
    label = mask_pil_to_long(label)

    return im, label


def processTestIm(data):
    (x, y) = data
    im = x.copy()

    norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])

    im_c1 = Image.fromarray(im[0]).resize((512, 512))
    im_c2 = Image.fromarray(im[1]).resize((512, 512))

    # Semantic labels must use nearest-neighbor interpolation.
    label = mask_to_pil(y).resize(
        (512, 512),
        resample=Image.Resampling.NEAREST,
    )

    im_c1s = [
        F.crop(im_c1, 0, 0, 256, 256),
        F.crop(im_c1, 0, 256, 256, 256),
        F.crop(im_c1, 256, 0, 256, 256),
        F.crop(im_c1, 256, 256, 256, 256),
    ]
    im_c2s = [
        F.crop(im_c2, 0, 0, 256, 256),
        F.crop(im_c2, 0, 256, 256, 256),
        F.crop(im_c2, 256, 0, 256, 256),
        F.crop(im_c2, 256, 256, 256, 256),
    ]
    labels = [
        F.crop(label, 0, 0, 256, 256),
        F.crop(label, 0, 256, 256, 256),
        F.crop(label, 256, 0, 256, 256),
        F.crop(label, 256, 256, 256, 256),
    ]

    ims = [
        torch.stack((
            transforms.ToTensor()(a).squeeze(),
            transforms.ToTensor()(b).squeeze(),
        ))
        for (a, b) in zip(im_c1s, im_c2s)
    ]
    ims = torch.stack([norm(im) for im in ims])

    labels = torch.stack([
        mask_pil_to_long(lab)
        for lab in labels
    ])

    return ims, labels

## 6. Build the training pairs

Two mechanisms are used, matching the historical experiment logic:

- CSV manifests: hand-labeled and permanent water.
- Directory pairing: S1 weak and S2 weak.

For weak data, pairing is done by chip key rather than blindly zipping sorted filenames. This is a safety check: it preserves the intended image↔label correspondence while failing loudly if the directory contents do not match.

In [ ]:
def read_csv_pairs(csv_path, image_dir, label_dir):
    pairs = []
    with open(csv_path, newline="") as f:
        for row in csv.reader(f):
            if not row:
                continue
            if len(row) < 2:
                raise ValueError(f"Malformed row in {csv_path}: {row}")
            image_path = Path(image_dir) / row[0]
            label_path = Path(label_dir) / row[1]
            pairs.append((image_path, label_path))

    missing = [(x, y) for x, y in pairs if not x.exists() or not y.exists()]
    if missing:
        preview = "\n".join(f"{x} | {y}" for x, y in missing[:10])
        raise FileNotFoundError(
            f"{len(missing)} CSV pairs are missing files. First examples:\n{preview}"
        )
    return pairs


WEAK_SUFFIXES = (
    "_S1Weak",
    "_S1OtsuLabelWeak",
    "_S2IndexLabelWeak",
)

def weak_chip_key(path):
    stem = Path(path).stem
    for suffix in WEAK_SUFFIXES:
        if stem.endswith(suffix):
            return stem[:-len(suffix)]
    raise ValueError(f"Unrecognized weak-label filename: {Path(path).name}")


def pair_weak_directories(image_dir, label_dir, exclude_bolivia=False):
    image_files = sorted(Path(image_dir).glob("*.tif"))
    label_files = sorted(Path(label_dir).glob("*.tif"))

    image_map = {}
    for p in image_files:
        key = weak_chip_key(p)
        if key in image_map:
            raise ValueError(f"Duplicate image key: {key}")
        image_map[key] = p

    label_map = {}
    for p in label_files:
        key = weak_chip_key(p)
        if key in label_map:
            raise ValueError(f"Duplicate label key: {key}")
        label_map[key] = p

    image_keys = set(image_map)
    label_keys = set(label_map)

    if image_keys != label_keys:
        only_images = sorted(image_keys - label_keys)[:20]
        only_labels = sorted(label_keys - image_keys)[:20]
        raise RuntimeError(
            "Weak image/label keys do not match.\n"
            f"Only in images: {only_images}\n"
            f"Only in labels: {only_labels}"
        )

    keys = sorted(image_keys)

    # Historical S2-weak behavior: Bolivia weak supervision was withheld.
    if exclude_bolivia:
        keys = [k for k in keys if "Bolivia" not in k]

    return [(image_map[k], label_map[k]) for k in keys]


kind = EXPERIMENTS[EXPERIMENT]["train_kind"]

if kind == "csv_hand":
    train_pairs = read_csv_pairs(FLOOD_TRAIN_CSV, S1_HAND_DIR, LABEL_HAND_DIR)
    valid_pairs = read_csv_pairs(FLOOD_VALID_CSV, S1_HAND_DIR, LABEL_HAND_DIR)
    input_scaling = "flood"

elif kind == "dir_s1_weak":
    train_pairs = pair_weak_directories(
        S1_WEAK_DIR,
        S1_WEAK_LABEL_DIR,
        exclude_bolivia=False,
    )
    valid_pairs = read_csv_pairs(FLOOD_VALID_CSV, S1_HAND_DIR, LABEL_HAND_DIR)
    input_scaling = "flood"

elif kind == "dir_s2_weak":
    train_pairs = pair_weak_directories(
        S1_WEAK_DIR,
        S2_WEAK_LABEL_DIR,
        exclude_bolivia=True,
    )
    valid_pairs = read_csv_pairs(FLOOD_VALID_CSV, S1_HAND_DIR, LABEL_HAND_DIR)
    input_scaling = "flood"

elif kind == "csv_perm":
    train_pairs = read_csv_pairs(PERM_TRAIN_CSV, S1_PERM_DIR, JRC_PERM_DIR)
    valid_pairs = read_csv_pairs(PERM_VALID_CSV, S1_PERM_DIR, JRC_PERM_DIR)

    # The historical permanent-water loader read S1Perm as stored,
    # unlike the flood/weak loader which clips dB to [-50, 1] and scales to [0, 1].
    input_scaling = "stored"

else:
    raise RuntimeError(kind)

# Common hand-labeled evaluation sets used for all four trained models.
test_pairs = read_csv_pairs(FLOOD_TEST_CSV, S1_HAND_DIR, LABEL_HAND_DIR)
bolivia_pairs = read_csv_pairs(FLOOD_BOLIVIA_CSV, S1_HAND_DIR, LABEL_HAND_DIR)

print("Training pairs:", len(train_pairs))
print("Validation pairs:", len(valid_pairs))
print("Common test pairs:", len(test_pairs))
print("Bolivia pairs:", len(bolivia_pairs))
print("Input scaling:", input_scaling)

print("\nFirst training pairs:")
for x, y in train_pairs[:5]:
    print(" ", x.name, "<->", y.name)
def preflight_validate_labels(pairs, split_name):
    seen = set()
    files_with_nonfinite = 0
    nonfinite_pixels_total = 0
    all_ignore_files = []

    for i, (_, label_path) in enumerate(pairs, start=1):
        with rasterio.open(label_path) as src:
            raw = src.read()

        clean, nonfinite_count = sanitize_label_array(
            raw,
            source=str(label_path),
            report_nonfinite=True,
            return_nonfinite_count=True,
        )

        if nonfinite_count:
            files_with_nonfinite += 1
            nonfinite_pixels_total += nonfinite_count

        values = np.unique(clean)
        seen.update(values.tolist())

        if np.all(clean == 255):
            all_ignore_files.append(str(label_path))

        if i % 500 == 0:
            print(f"  {split_name}: checked {i}/{len(pairs)} masks")

    print(
        f"{split_name}: {len(pairs)} masks OK; "
        f"target values after canonicalization = {sorted(seen)}"
    )

    if files_with_nonfinite:
        print(
            f"{split_name}: {files_with_nonfinite} file(s) contained "
            f"{nonfinite_pixels_total} NaN/Inf target pixels in total; "
            "those pixels are ignored (255)."
        )

    if all_ignore_files:
        print(
            f"WARNING: {split_name}: {len(all_ignore_files)} mask(s) contain "
            "only ignored pixels after canonicalization."
        )
        for p in all_ignore_files[:10]:
            print("  all-ignore:", p)

    return {
        "files": len(pairs),
        "values": sorted(seen),
        "files_with_nonfinite": files_with_nonfinite,
        "nonfinite_pixels": nonfinite_pixels_total,
        "all_ignore_files": all_ignore_files,
    }


if PREFLIGHT_VALIDATE_LABELS:
    print("\nCPU target preflight:")
    preflight_summary = {
        "train": preflight_validate_labels(train_pairs, "train"),
        "validation": preflight_validate_labels(valid_pairs, "validation"),
        "common_test": preflight_validate_labels(test_pairs, "common test"),
        "bolivia_test": preflight_validate_labels(bolivia_pairs, "Bolivia test"),
    }
    print("All target masks passed preflight.")

Training pairs: 488
Validation pairs: 163
Common test pairs: 90
Bolivia pairs: 15
Input scaling: stored

First training pairs:
  sentinel_4_673_high_density_120.94158929849634_40.69811721457736.tif <-> water_4_673_high_density_120.94158929849634_40.69811721457736.tif
  sentinel_8_390_rural_-95.59962173245162_35.3399454521294.tif <-> water_8_390_rural_-95.59962173245162_35.3399454521294.tif
  sentinel_13_106_rural_51.23629289778179_11.653419152176511.tif <-> water_13_106_rural_51.23629289778179_11.653419152176511.tif
  sentinel_2_32_inhabited_47.48036978495905_-14.710632343631648.tif <-> water_2_521_inhabited_-102.92399843160813_20.207531198953358.tif
  sentinel_1_159_rural_169.45186427646658_-19.517775152501088.tif <-> water_1_159_rural_169.45186427646658_-19.517775152501088.tif

CPU target preflight:
train: 488 masks OK; target values after canonicalization = [0, 1, 255]
train: 42 file(s) contained 3801 NaN/Inf target pixels in total; those pixels are ignored (255).
validation: 163 ma

## 7. Lazy TIFF dataset

TIFFs are read from the local Windows disk only when a sample is requested.

This avoids loading the entire 4,384-chip weak dataset into RAM and does not change the numerical preprocessing applied to each sample.

In [ ]:
class Sen1FloodsPathDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, preprocess_func, input_scaling):
        self.pairs = list(pairs)
        self.preprocess_func = preprocess_func
        self.input_scaling = input_scaling

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        im_path, label_path = self.pairs[index]

        with rasterio.open(im_path) as src:
            arr_x = src.read()

        with rasterio.open(label_path) as src:
            arr_y = src.read()

        arr_x = np.nan_to_num(arr_x)

        # Current/released flood loader behavior.
        if self.input_scaling == "flood":
            arr_x = np.clip(arr_x, -50, 1)
            arr_x = (arr_x + 50) / 51

        elif self.input_scaling == "stored":
            # Historical permanent-water loader behavior.
            pass

        else:
            raise ValueError(self.input_scaling)

        arr_y = sanitize_label_array(
            arr_y,
            source=str(label_path),
            report_nonfinite=False,
        )
        arr_y = arr_y[None, ...]

        try:
            return self.preprocess_func((arr_x, arr_y))
        except Exception as exc:
            raise RuntimeError(
                f"Failed preprocessing image={im_path} label={label_path}"
            ) from exc

In [ ]:
# Training and validation loaders preserve the released batch sizes and shuffle settings.
train_dataset = Sen1FloodsPathDataset(
    train_pairs,
    processAndAugment,
    input_scaling=input_scaling,
)

valid_dataset = Sen1FloodsPathDataset(
    valid_pairs,
    processTestIm,
    input_scaling=input_scaling if EXPERIMENT == "permanent_water" else "flood",
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    sampler=None,
    batch_sampler=None,
    num_workers=0,
    collate_fn=None,
    pin_memory=True,
    drop_last=False,
    timeout=0,
    worker_init_fn=None,
)

valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=True,
    sampler=None,
    batch_sampler=None,
    num_workers=0,
    collate_fn=lambda x: (
        torch.cat([a[0] for a in x], 0),
        torch.cat([a[1] for a in x], 0),
    ),
    pin_memory=True,
    drop_last=False,
    timeout=0,
    worker_init_fn=None,
)

# Common hand-labeled test sets always use flood preprocessing.
test_dataset = Sen1FloodsPathDataset(test_pairs, processTestIm, input_scaling="flood")
bolivia_dataset = Sen1FloodsPathDataset(bolivia_pairs, processTestIm, input_scaling="flood")

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda x: (
        torch.cat([a[0] for a in x], 0),
        torch.cat([a[1] for a in x], 0),
    ),
    pin_memory=True,
)

bolivia_loader = torch.utils.data.DataLoader(
    bolivia_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda x: (
        torch.cat([a[0] for a in x], 0),
        torch.cat([a[1] for a in x], 0),
    ),
    pin_memory=True,
)

print("train batches:", len(train_loader))
print("validation batches:", len(valid_loader))

train batches: 31
validation batches: 41


## 8. FCN-ResNet50, loss, optimizer, scheduler

The training configuration below follows the released notebook.

A compatibility fallback is included for newer torchvision versions where the old `pretrained=` arguments may have been removed; it constructs the same uninitialized FCN-ResNet50 architecture.

In [ ]:
import torch.nn as nn
import torchvision.models as models

try:
    net = models.segmentation.fcn_resnet50(
        pretrained=False,
        num_classes=2,
        pretrained_backbone=False,
    )
except TypeError:
    net = models.segmentation.fcn_resnet50(
        weights=None,
        weights_backbone=None,
        num_classes=2,
    )

net.backbone.conv1 = nn.Conv2d(
    2, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False,
)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1, 8]).float().cuda(),
    ignore_index=255,
)

optimizer = torch.optim.AdamW(net.parameters(), lr=LR)

# Intentionally preserved from released code.
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    len(train_loader) * 10,
    T_mult=2,
    eta_min=0,
    last_epoch=-1,
)

def convertBNtoGN(module, num_groups=16):
    if isinstance(module, torch.nn.modules.batchnorm.BatchNorm2d):
        return nn.GroupNorm(
            num_groups,
            module.num_features,
            eps=module.eps,
            affine=module.affine,
        )
        if module.affine:
            mod.weight.data = module.weight.data.clone().detach()
            mod.bias.data = module.bias.data.clone().detach()

    for name, child in module.named_children():
        module.add_module(name, convertBNtoGN(child, num_groups=num_groups))

    return module

net = convertBNtoGN(net)
net = net.cuda()

print("Model ready on:", next(net.parameters()).device)

d:\UNI\THESIS\.venv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\UNI\THESIS\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
d:\UNI\THESIS\.venv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained_backbone' is deprecated since 0.13 and may be removed in the future, please use 'weights_backbone' instead.
  warnings.warn(
d:\UNI\THESIS\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights_backbone' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing 

Model ready on: cuda:0


## 9. Released metrics

In [ ]:
def validate_target_tensor(target, context="target"):
    """Validate target IDs on CPU before copying the target to CUDA."""
    t = target.detach().cpu()

    if t.is_floating_point():
        rounded = t.round()
        if not torch.allclose(t, rounded, rtol=0, atol=1e-6):
            vals = torch.unique(t)
            raise ValueError(
                f"{context}: non-integer target values. "
                f"First unique values: {vals[:30].tolist()}"
            )
        t = rounded

    vals = torch.unique(t.to(torch.int64)).tolist()
    invalid = sorted(int(v) for v in vals if int(v) not in VALID_TARGET_VALUES)

    if invalid:
        raise ValueError(
            f"{context}: invalid target IDs {invalid}. "
            "Allowed IDs: 0, 1, 255."
        )

    return vals


def target_has_supervision(target):
    return bool(torch.any(target.detach().cpu() != 255).item())


def _valid_flattened(output, target):
    pred = torch.argmax(output, dim=1).flatten()
    target = target.flatten().to(pred.device)
    keep = target.ne(255)
    return pred.masked_select(keep), target.masked_select(keep)


def computeIOU(output, target):
    output, target = _valid_flattened(output, target)

    if target.numel() == 0:
        return torch.tensor(0.0, device=output.device)

    intersection = torch.sum(output * target)
    union = torch.sum(target) + torch.sum(output) - intersection
    return (intersection + 1e-7) / (union + 1e-7)


def computeAccuracy(output, target):
    output, target = _valid_flattened(output, target)

    if target.numel() == 0:
        return torch.tensor(0.0, device=output.device)

    correct = torch.sum(output.eq(target))
    return correct.float() / target.numel()


def truePositives(output, target):
    output, target = _valid_flattened(output, target)
    return torch.sum(output * target)


def trueNegatives(output, target):
    output, target = _valid_flattened(output, target)
    return torch.sum((output == 0) * (target == 0))


def falsePositives(output, target):
    output, target = _valid_flattened(output, target)
    return torch.sum((output == 1) * (target == 0))


def falseNegatives(output, target):
    output, target = _valid_flattened(output, target)
    return torch.sum((output == 0) * (target == 1))

## 10. Released training / validation computation

In [ ]:
training_losses = []
training_accuracies = []
training_ious = []

valid_losses = []
valid_accuracies = []
valid_ious = []

running_loss = 0
running_iou = 0
running_count = 0
running_accuracy = 0

max_valid_iou = -float("inf")
best_checkpoint = None


def train_loop(inputs, labels, net, optimizer, scheduler):
    global running_loss
    global running_iou
    global running_count
    global running_accuracy

    # Fail here on CPU with a readable error, not later inside a CUDA kernel.
    validate_target_tensor(labels, context="training batch")

    if not target_has_supervision(labels):
        print("WARNING: skipping all-ignore training batch.")
        return False

    optimizer.zero_grad()

    inputs = inputs.cuda(non_blocking=True)
    labels = labels.long().cuda(non_blocking=True)

    outputs = net(inputs)
    loss = criterion(outputs["out"], labels)

    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite training loss: {loss.item()}")

    loss.backward()
    optimizer.step()
    scheduler.step()

    running_loss += loss.detach()
    running_iou += computeIOU(outputs["out"], labels).detach()
    running_accuracy += computeAccuracy(outputs["out"], labels).detach()
    running_count += 1

    return True


def _rng_state_for_checkpoint():
    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state().cpu(),
    }
    if torch.cuda.is_available():
        state["torch_cuda_rng_state_all"] = [
            s.cpu() for s in torch.cuda.get_rng_state_all()
        ]
    return state


def save_resume_checkpoint(completed_epoch, history):
    checkpoint = {
        "completed_epoch": completed_epoch,
        "model_state_dict": net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "max_valid_iou": (
            max_valid_iou.item()
            if isinstance(max_valid_iou, torch.Tensor)
            else float(max_valid_iou)
        ),
        "best_checkpoint": str(best_checkpoint) if best_checkpoint is not None else None,
        "training_losses": training_losses,
        "training_accuracies": training_accuracies,
        "training_ious": training_ious,
        "valid_losses": valid_losses,
        "valid_accuracies": valid_accuracies,
        "valid_ious": valid_ious,
        "history": history,
        **_rng_state_for_checkpoint(),
    }

    tmp_path = RESUME_CHECKPOINT.with_suffix(".tmp")
    torch.save(checkpoint, tmp_path)
    os.replace(tmp_path, RESUME_CHECKPOINT)

    print("Resume checkpoint saved:", RESUME_CHECKPOINT)


def _cpu_byte_rng_state(state, name):
    if not isinstance(state, torch.Tensor):
        raise TypeError(f"{name} is {type(state)}, expected torch.Tensor")
    return state.detach().cpu().to(dtype=torch.uint8)


def load_resume_checkpoint():
    global max_valid_iou
    global best_checkpoint
    global training_losses, training_accuracies, training_ious
    global valid_losses, valid_accuracies, valid_ious

    # CPU-first loading prevents the torch.set_rng_state CUDA tensor bug.
    checkpoint = torch.load(
        RESUME_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    net.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    max_valid_iou = checkpoint["max_valid_iou"]
    saved_best = checkpoint.get("best_checkpoint")
    best_checkpoint = Path(saved_best) if saved_best else None

    training_losses = checkpoint.get("training_losses", [])
    training_accuracies = checkpoint.get("training_accuracies", [])
    training_ious = checkpoint.get("training_ious", [])
    valid_losses = checkpoint.get("valid_losses", [])
    valid_accuracies = checkpoint.get("valid_accuracies", [])
    valid_ious = checkpoint.get("valid_ious", [])

    if "python_random_state" in checkpoint:
        random.setstate(checkpoint["python_random_state"])

    if "numpy_random_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_random_state"])

    if "torch_rng_state" in checkpoint:
        torch.set_rng_state(
            _cpu_byte_rng_state(checkpoint["torch_rng_state"], "torch_rng_state")
        )

    if torch.cuda.is_available() and "torch_cuda_rng_state_all" in checkpoint:
        cuda_states = [
            _cpu_byte_rng_state(s, f"torch_cuda_rng_state_all[{i}]")
            for i, s in enumerate(checkpoint["torch_cuda_rng_state_all"])
        ]
        torch.cuda.set_rng_state_all(cuda_states)

    completed_epoch = int(checkpoint["completed_epoch"])
    history = checkpoint.get("history", [])

    print(f"Resumed from completed epoch {completed_epoch}.")
    print("Best validation IoU so far:", max_valid_iou)
    print("Best checkpoint so far:", best_checkpoint)

    return completed_epoch, history

In [ ]:
def validation_loop(validation_data_loader, net, cur_epoch):
    global running_loss
    global running_iou
    global running_count
    global running_accuracy
    global max_valid_iou
    global best_checkpoint

    net = net.eval()

    count = 0
    iou = 0
    loss = 0
    accuracy = 0
    skipped_all_ignore = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(validation_data_loader):
            validate_target_tensor(
                labels,
                context=f"validation batch {batch_idx}",
            )

            if not target_has_supervision(labels):
                skipped_all_ignore += 1
                continue

            images = images.cuda(non_blocking=True)
            labels = labels.long().cuda(non_blocking=True)

            outputs = net(images)
            valid_loss = criterion(outputs["out"], labels)

            if not torch.isfinite(valid_loss):
                raise FloatingPointError(
                    f"Non-finite validation loss in batch {batch_idx}: "
                    f"{valid_loss.item()}"
                )

            valid_iou = computeIOU(outputs["out"], labels)
            valid_accuracy = computeAccuracy(outputs["out"], labels)

            iou += valid_iou
            loss += valid_loss
            accuracy += valid_accuracy
            count += 1

    if count == 0:
        raise RuntimeError("Validation set contains no supervised pixels.")

    if skipped_all_ignore:
        print(
            f"WARNING: skipped {skipped_all_ignore} all-ignore validation batch(es)."
        )

    if running_count == 0:
        raise RuntimeError("No supervised training batch was processed this epoch.")

    iou = iou / count
    loss = loss / count
    accuracy = accuracy / count

    if iou > max_valid_iou:
        max_valid_iou = iou
        save_path = CHECKPOINT_DIR / (
            f"{RUNNAME}_epoch{cur_epoch:03d}_valIoU{iou.item():.6f}.cp"
        )
        torch.save(net.state_dict(), save_path)
        best_checkpoint = save_path
        print("model saved at", save_path)

    print("Training Loss:", running_loss / running_count)
    print("Training IOU:", running_iou / running_count)
    print("Training Accuracy:", running_accuracy / running_count)
    print("Validation Loss:", loss)
    print("Validation IOU:", iou)
    print("Validation Accuracy:", accuracy)

    training_losses.append((running_loss / running_count).item())
    training_accuracies.append((running_accuracy / running_count).item())
    training_ious.append((running_iou / running_count).item())

    valid_losses.append(loss.item())
    valid_accuracies.append(accuracy.item())
    valid_ious.append(iou.item())

    return {
        "epoch": cur_epoch,
        "training_loss": training_losses[-1],
        "training_iou": training_ious[-1],
        "training_accuracy": training_accuracies[-1],
        "validation_loss": valid_losses[-1],
        "validation_iou": valid_ious[-1],
        "validation_accuracy": valid_accuracies[-1],
    }

In [ ]:
from tqdm.notebook import tqdm

def train_epoch(net, optimizer, scheduler, train_iter):
    for (inputs, labels) in tqdm(train_iter, leave=False):
        # Keep targets on CPU until train_loop validates their class IDs.
        train_loop(
            inputs,
            labels,
            net,
            optimizer,
            scheduler,
        )


def train_validation_loop(
    net,
    optimizer,
    scheduler,
    train_loader,
    valid_loader,
    num_epochs,
    cur_epoch,
):
    global running_loss
    global running_iou
    global running_count
    global running_accuracy

    net = net.train()

    running_loss = 0
    running_iou = 0
    running_count = 0
    running_accuracy = 0

    for _ in tqdm(range(num_epochs), leave=False):
        train_iter = iter(train_loader)
        train_epoch(net, optimizer, scheduler, train_iter)

    print("\nCurrent Epoch:", cur_epoch)
    return validation_loop(iter(valid_loader), net, cur_epoch)

## 11. Train

This executes the experiment-specific published epoch count and writes the best-validation-IoU checkpoint directly to Drive.

The notebook intentionally does **not** set a random seed because the released code did not specify one.

In [ ]:
# ---------------------------------------------------------------
# TRAIN / RESUME
# ---------------------------------------------------------------
if RESUME_IF_AVAILABLE and RESUME_CHECKPOINT.exists():
    completed, history = load_resume_checkpoint()
else:
    completed = 0
    history = []
    print("Starting a new run from epoch 1.")

if completed >= EPOCHS:
    print(f"Training already completed: {completed}/{EPOCHS} epochs.")
else:
    while completed < EPOCHS:
        block = min(EPOCHS_PER_UPDATE, EPOCHS - completed)
        completed += block

        record = train_validation_loop(
            net,
            optimizer,
            scheduler,
            train_loader,
            valid_loader,
            block,
            completed,
        )
        history.append(record)

        # Human-readable history.
        history_path = RESULTS_DIR / "training_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        # Full training state. Safe to use after a reboot/kernel restart.
        save_resume_checkpoint(completed, history)

print("\nTraining state:")
print("Completed epochs:", completed, "/", EPOCHS)
print("Best validation IoU:", max_valid_iou)
print("Best checkpoint:", best_checkpoint)

Starting a new run from epoch 1.


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/31 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 12. Common hand-labeled test sets

After training, this loads the best-validation-IoU model and evaluates it on:
- the normal hand-labeled test split
- the separately held-out Bolivia hand-labeled split

Results are written to the local Windows disk under the experiment's `results` folder.

For exact reconstruction of every paper table (including permanent-vs-flood subcategories and Otsu), the historical `Test_Models.ipynb` still needs to be adapted separately.

In [ ]:
def test_loop(test_data_loader, net, name="test"):
    net = net.eval()

    count = 0
    iou = 0
    accuracy = 0
    skipped_all_ignore = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(test_data_loader, leave=False)):
            validate_target_tensor(labels, context=f"{name} batch {batch_idx}")

            if not target_has_supervision(labels):
                skipped_all_ignore += 1
                continue

            images = images.cuda(non_blocking=True)
            labels = labels.long().cuda(non_blocking=True)

            outputs = net(images)
            iou += computeIOU(outputs["out"], labels)
            accuracy += computeAccuracy(outputs["out"], labels)
            count += 1

    if count == 0:
        raise RuntimeError(f"{name}: no supervised pixels available.")

    if skipped_all_ignore:
        print(
            f"WARNING: {name} skipped {skipped_all_ignore} all-ignore batch(es)."
        )

    mean_iou = (iou / count).item()
    mean_accuracy = (accuracy / count).item()

    print(f"{name} IOU:", mean_iou)
    print(f"{name} Accuracy:", mean_accuracy)

    return {
        "name": name,
        "iou": mean_iou,
        "accuracy": mean_accuracy,
    }


if best_checkpoint is None:
    raise RuntimeError("No checkpoint was saved. Train the model first.")

print("Loading:", best_checkpoint)
net.load_state_dict(torch.load(best_checkpoint, map_location="cuda"))

test_results = [
    test_loop(test_loader, net, "hand_labeled_test"),
    test_loop(bolivia_loader, net, "bolivia_test"),
]

with open(RESULTS_DIR / "common_test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("Saved:", RESULTS_DIR / "common_test_results.json")

## 13. Save environment / run configuration

In [ ]:
run_info = {
    "experiment": EXPERIMENT,
    "dataset_root": str(DATASET_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "epochs": EPOCHS,
    "epochs_per_update": EPOCHS_PER_UPDATE,
    "learning_rate": LR,
    "batch_size": 16,
    "validation_batch_size": 4,
    "scheduler_T0_batches": len(train_loader) * 10,
    "scheduler_Tmult": 2,
    "python": sys.version,
    "platform": platform.platform(),
    "hostname": platform.node(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "train_pairs": len(train_pairs),
    "valid_pairs": len(valid_pairs),
    "test_pairs": len(test_pairs),
    "bolivia_pairs": len(bolivia_pairs),
    "input_scaling": input_scaling,
    "resume_checkpoint": str(RESUME_CHECKPOINT),
    "best_checkpoint": str(best_checkpoint) if best_checkpoint else None,
    "best_validation_iou": (
        float(max_valid_iou.item())
        if isinstance(max_valid_iou, torch.Tensor)
        else float(max_valid_iou)
    ),
}

with open(LOG_DIR / "run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print(json.dumps(run_info, indent=2))
print("\nSaved:", LOG_DIR / "run_info.json")